# Temperature Scaling Calibration — T8 MC Dropout

**Goal:** Post-hoc calibration of MC Dropout uncertainty estimates using temperature scaling.  
**Input:** Pre-computed `mc_dropout_full_100graphs_mc30.npz` (already on Drive — no GPU needed).  
**Output saved to Drive:**
```
point_net_transf_gat_8th_trial_lower_dropout/uq_results/
  temperature_scaling_results.json   ← NEW: verified numbers
  fig_temp_scaling_4panel.png        ← NEW: 4-panel diagnostic figure
  fig_temp_scaling_reliability.png   ← NEW: before/after reliability diagram
```

> **No GPU required** — runs on CPU in < 2 minutes.

## Cell 1 — Mount Drive + Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

print('Imports OK')

## Cell 2 — Config + Path Check

In [ ]:
DRIVE_BASE = '/content/drive/MyDrive/data/TR-C_Benchmarks'
T8_UQ_DIR  = f'{DRIVE_BASE}/point_net_transf_gat_8th_trial_lower_dropout/uq_results'

NPZ_PATH   = f'{T8_UQ_DIR}/mc_dropout_full_100graphs_mc30.npz'
OUT_JSON   = f'{T8_UQ_DIR}/temperature_scaling_results.json'
OUT_FIG4   = f'{T8_UQ_DIR}/fig_temp_scaling_4panel.png'
OUT_FIGREL = f'{T8_UQ_DIR}/fig_temp_scaling_reliability.png'

print(f'NPZ exists : {os.path.exists(NPZ_PATH)}')
print(f'Output dir : {T8_UQ_DIR}')

if not os.path.exists(NPZ_PATH):
    raise FileNotFoundError(f'NPZ not found: {NPZ_PATH}')
print('Ready.')

## Cell 3 — Core Functions

In [ ]:
def compute_ece(uncertainties, errors, n_bins=10):
    """
    Regression ECE: bin by uncertainty percentile.
    For each bin, measure fraction with |error| < sigma (expected 68.3%).
    ECE = weighted mean absolute deviation from 0.683 target.
    """
    bin_edges = np.percentile(uncertainties, np.linspace(0, 100, n_bins + 1))
    bin_edges = np.unique(bin_edges)

    ece = 0.0
    total = len(uncertainties)
    calib_data = []

    for i in range(len(bin_edges) - 1):
        if i == len(bin_edges) - 2:
            mask = (uncertainties >= bin_edges[i]) & (uncertainties <= bin_edges[i+1])
        else:
            mask = (uncertainties >= bin_edges[i]) & (uncertainties < bin_edges[i+1])
        if mask.sum() == 0:
            continue
        obs_cov = float(np.mean(errors[mask] < uncertainties[mask]))
        exp_cov = 0.683
        w = mask.sum() / total
        ece += w * abs(obs_cov - exp_cov)
        calib_data.append({
            'bin_center'      : float((bin_edges[i] + bin_edges[i+1]) / 2),
            'observed_coverage': obs_cov,
            'expected_coverage': exp_cov,
            'n_samples'       : int(mask.sum()),
        })
    return float(ece), calib_data


def coverage_at_sigma(uncertainties, errors, sigma_level):
    return float(np.mean(errors < sigma_level * uncertainties))


def find_optimal_temperature(val_unc, val_err):
    """Grid search + bounded optimization on validation set."""
    T_grid  = np.logspace(-1, 2, 80)
    ece_grid = [compute_ece(val_unc * T, val_err)[0] for T in T_grid]
    T_init  = T_grid[int(np.argmin(ece_grid))]
    print(f'  Grid best : T={T_init:.4f}  ECE={min(ece_grid):.5f}')

    res = minimize_scalar(
        lambda T: compute_ece(val_unc * T, val_err)[0],
        bounds=(T_init * 0.5, T_init * 2.0),
        method='bounded'
    )
    T_opt = float(res.x)
    print(f'  Optimised : T={T_opt:.4f}  ECE={res.fun:.5f}')
    return T_opt, T_grid, np.array(ece_grid)


print('Functions defined OK')

## Cell 4 — Run Temperature Scaling + Save JSON

In [ ]:
print('=' * 65)
print('  TEMPERATURE SCALING CALIBRATION — T8 MC Dropout')
print('=' * 65)

# ── Load NPZ ─────────────────────────────────────────────────────────────
print('\n[1/4] Loading MC Dropout NPZ...')
data = np.load(NPZ_PATH)
preds  = data['predictions'].flatten().astype(np.float64)
sigmas = data['uncertainties'].flatten().astype(np.float64)
tgts   = data['targets'].flatten().astype(np.float64)
errors = np.abs(preds - tgts)
print(f'  Loaded {len(preds):,} node predictions')
print(f'  sigma range : [{sigmas.min():.4f}, {sigmas.max():.4f}]')
print(f'  error range : [{errors.min():.4f}, {errors.max():.4f}]')

# ── Train / Val / Test split (node-level, same seed as original script) ──
print('\n[2/4] Splitting into val (30%) / test (70%)...')
np.random.seed(42)
idx = np.random.permutation(len(preds))
n_val  = int(0.30 * len(preds))
val_idx, test_idx = idx[:n_val], idx[n_val:]

val_unc,  val_err  = sigmas[val_idx],  errors[val_idx]
test_unc, test_err = sigmas[test_idx], errors[test_idx]
print(f'  Val  : {len(val_idx):,} nodes')
print(f'  Test : {len(test_idx):,} nodes')

# ── Baseline ECE (T=1) ───────────────────────────────────────────────────
ece_raw, _  = compute_ece(test_unc, test_err)
cov1_raw    = coverage_at_sigma(test_unc, test_err, 1)
cov2_raw    = coverage_at_sigma(test_unc, test_err, 2)
cov3_raw    = coverage_at_sigma(test_unc, test_err, 3)
print(f'\n  Baseline (T=1.0):')
print(f'    ECE = {ece_raw:.4f}')
print(f'    1σ coverage = {cov1_raw:.3f}  (expected 0.683)')
print(f'    2σ coverage = {cov2_raw:.3f}  (expected 0.954)')
print(f'    3σ coverage = {cov3_raw:.3f}  (expected 0.997)')

# ── Find optimal T on validation set ─────────────────────────────────────
print('\n[3/4] Optimising temperature on validation set...')
t0 = time.time()
T_opt, T_grid, ece_grid = find_optimal_temperature(val_unc, val_err)
print(f'  Done in {time.time()-t0:.1f}s')

# ── Calibrated evaluation on test set ────────────────────────────────────
ece_cal, calib_bins = compute_ece(test_unc * T_opt, test_err)
cov1_cal = coverage_at_sigma(test_unc, test_err, T_opt * 1)
cov2_cal = coverage_at_sigma(test_unc, test_err, T_opt * 2)
cov3_cal = coverage_at_sigma(test_unc, test_err, T_opt * 3)
improvement_pct = (ece_raw - ece_cal) / ece_raw * 100

print(f'\n  Calibrated (T={T_opt:.4f}):')
print(f'    ECE = {ece_cal:.4f}  (was {ece_raw:.4f}, -{improvement_pct:.1f}%)')
print(f'    1σ coverage = {cov1_cal:.3f}  (expected 0.683)')
print(f'    2σ coverage = {cov2_cal:.3f}  (expected 0.954)')
print(f'    3σ coverage = {cov3_cal:.3f}  (expected 0.997)')

# ── Effective k95 after scaling ──────────────────────────────────────────
k95_raw = float(np.percentile(errors / np.clip(sigmas,         1e-10, None), 95))
k95_cal = float(np.percentile(errors / np.clip(sigmas * T_opt, 1e-10, None), 95))
print(f'\n  k95 (raw)       = {k95_raw:.2f}  (ideal Gaussian: 1.96)')
print(f'  k95 (calibrated)= {k95_cal:.2f}')

# ── Save JSON ────────────────────────────────────────────────────────────
print('\n[4/4] Saving JSON...')
results = {
    'experiment'        : 'temperature_scaling_calibration',
    'trial'             : 8,
    'verified_at'       : time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime()),
    'method'            : 'temperature_scaling',
    'n_total_nodes'     : int(len(preds)),
    'n_val_nodes'       : int(len(val_idx)),
    'n_test_nodes'      : int(len(test_idx)),
    'val_fraction'      : 0.30,
    'split_seed'        : 42,
    'optimal_temperature': T_opt,
    'before_calibration': {
        'ece'           : ece_raw,
        'coverage_1sig' : cov1_raw,
        'coverage_2sig' : cov2_raw,
        'coverage_3sig' : cov3_raw,
        'k95'           : k95_raw,
    },
    'after_calibration': {
        'ece'           : ece_cal,
        'ece_improvement_pct': improvement_pct,
        'coverage_1sig' : cov1_cal,
        'coverage_2sig' : cov2_cal,
        'coverage_3sig' : cov3_cal,
        'k95'           : k95_cal,
    },
    'calibration_bins'  : calib_bins,
    'notes': (
        'T optimised on val (30% of node-level predictions, seed=42). '
        'ECE definition: bin by sigma percentile, measure 1-sigma coverage per bin, '
        'weighted MAD from expected 0.683. '
        'k95 = 95th percentile of |error|/sigma ratio.'
    ),
}
with open(OUT_JSON, 'w') as f:
    json.dump(results, f, indent=2)
print(f'  Saved: {OUT_JSON}')

print(f'''
  ┌──────────────────────────────────────────────────┐
  │  TEMPERATURE SCALING SUMMARY                     │
  │  Optimal T        = {T_opt:>8.4f}                   │
  │  ECE before       = {ece_raw:>8.4f}                   │
  │  ECE after        = {ece_cal:>8.4f}  ({improvement_pct:+.1f}%)          │
  │  k95 before/after = {k95_raw:>5.2f} / {k95_cal:<5.2f}               │
  └──────────────────────────────────────────────────┘
''')

## Cell 5 — Generate & Save Figures

In [ ]:
plt.rcParams.update({
    'font.family'       : 'DejaVu Sans',
    'font.size'         : 10,
    'axes.titlesize'    : 11,
    'axes.labelsize'    : 10,
    'xtick.labelsize'   : 9,
    'ytick.labelsize'   : 9,
    'legend.fontsize'   : 9,
    'figure.dpi'        : 150,
    'savefig.dpi'       : 300,
    'savefig.bbox'      : 'tight',
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
})

C_RAW = '#e07b39'
C_CAL = '#2c7bb6'
C_PER = '#2d2d2d'

# ─────────────────────────────────────────────────────────────────────────
# FIG 1: 4-panel diagnostic
# ─────────────────────────────────────────────────────────────────────────
print('Generating Fig 1: 4-panel diagnostic...')
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Panel 1 — T vs ECE curve
ax = axes[0, 0]
ax.semilogx(T_grid, ece_grid, lw=2, color=C_PER)
ax.axvline(T_opt, color=C_CAL, ls='--', lw=2, label=f'Optimal T = {T_opt:.2f}')
ax.axvline(1.0,   color=C_RAW, ls=':',  lw=2, label='Original T = 1.0')
ax.set_xlabel('Temperature T  (log scale)')
ax.set_ylabel('Expected Calibration Error (ECE)')
ax.set_title('Temperature Optimisation', fontweight='bold')
ax.legend()

# Panel 2 — coverage bar chart (1σ / 2σ / 3σ)
ax = axes[0, 1]
sigma_labels  = ['1σ (68.3%)', '2σ (95.4%)', '3σ (99.7%)']
expected_covs = [0.683, 0.954, 0.997]
raw_covs      = [cov1_raw, cov2_raw, cov3_raw]
cal_covs      = [cov1_cal, cov2_cal, cov3_cal]

x = np.arange(3)
w = 0.25
b1 = ax.bar(x - w,  expected_covs, w, label='Expected', color='#d0d0d0', edgecolor='black', lw=0.8)
b2 = ax.bar(x,      raw_covs,      w, label=f'Before (T=1.0)', color=C_RAW, edgecolor='black', lw=0.8)
b3 = ax.bar(x + w,  cal_covs,      w, label=f'After (T={T_opt:.2f})',  color=C_CAL, edgecolor='black', lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(sigma_labels)
ax.set_ylabel('Coverage probability')
ax.set_ylim(0, 1.15)
ax.set_title('Coverage Before vs. After', fontweight='bold')
ax.legend()
for bars in [b1, b2, b3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)

# Panel 3 — Reliability diagram BEFORE
def reliability_diagram(ax, unc, err, label, color):
    n_bins   = 10
    edges    = np.percentile(unc, np.linspace(0, 100, n_bins + 1))
    centers, covs = [], []
    for i in range(len(edges) - 1):
        mask = (unc >= edges[i]) & (unc < edges[i+1] if i < len(edges)-2 else unc <= edges[i+1])
        if mask.sum() == 0: continue
        centers.append(float(unc[mask].mean()))
        covs.append(float(np.mean(err[mask] < unc[mask])))
    ax.axhline(0.683, color=C_PER, ls='--', lw=1.5, label='Perfect (68.3%)')
    ax.scatter(centers, covs, s=80, color=color, zorder=5)
    ax.plot(centers, covs, '-', color=color, lw=2, alpha=0.8, label=label)
    ece_val, _ = compute_ece(unc, err)
    ax.text(0.95, 0.08, f'ECE = {ece_val:.4f}', transform=ax.transAxes,
            ha='right', va='bottom', fontsize=10, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.9, edgecolor='none'))
    ax.set_ylabel('Observed 1σ coverage')
    ax.set_ylim(0, 1.0)
    ax.legend(loc='lower right')

ax = axes[1, 0]
reliability_diagram(ax, test_unc, test_err, 'Before (T=1.0)', C_RAW)
ax.set_xlabel('Mean σ (veh/h)')
ax.set_title('Reliability Diagram — Before', fontweight='bold')

# Panel 4 — Reliability diagram AFTER
ax = axes[1, 1]
reliability_diagram(ax, test_unc * T_opt, test_err, f'After (T={T_opt:.2f})', C_CAL)
ax.set_xlabel(f'Mean σ × T={T_opt:.2f}  (veh/h)')
ax.set_title('Reliability Diagram — After', fontweight='bold')

fig.suptitle(
    f'Temperature Scaling Calibration — T8 MC Dropout  '
    f'(T={T_opt:.2f}, ECE: {ece_raw:.3f}→{ece_cal:.3f}, −{improvement_pct:.0f}%)',
    fontsize=12, fontweight='bold', y=1.01
)
fig.tight_layout()
fig.savefig(OUT_FIG4)
plt.close(fig)
print(f'  Saved: {OUT_FIG4}')

# ─────────────────────────────────────────────────────────────────────────
# FIG 2: Thesis-ready before/after reliability (1×2)
# ─────────────────────────────────────────────────────────────────────────
print('Generating Fig 2: Thesis reliability diagram...')
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

ax = axes[0]
reliability_diagram(ax, test_unc, test_err, 'T=1.0 (uncalibrated)', C_RAW)
ax.set_xlabel('Predicted uncertainty σ (veh/h)')
ax.set_title(f'Before Calibration  (ECE = {ece_raw:.3f})', fontweight='bold')

ax = axes[1]
reliability_diagram(ax, test_unc * T_opt, test_err, f'T={T_opt:.2f} (calibrated)', C_CAL)
ax.set_xlabel(f'Calibrated uncertainty σ × {T_opt:.2f}  (veh/h)')
ax.set_title(f'After Calibration  (ECE = {ece_cal:.3f})', fontweight='bold')

fig.suptitle(
    f'Temperature T = {T_opt:.2f}  |  ECE improvement: −{improvement_pct:.0f}%  |  '
    f'k₉₅: {k95_raw:.2f} → {k95_cal:.2f}',
    y=1.02, fontsize=11
)
fig.tight_layout()
fig.savefig(OUT_FIGREL)
plt.close(fig)
print(f'  Saved: {OUT_FIGREL}')

print()
print('All done. Files saved to Drive:')
print(f'  {OUT_JSON}')
print(f'  {OUT_FIG4}')
print(f'  {OUT_FIGREL}')